In [ ]:
%pip install neo4j python-dotenv langchain langchain-groq langchain-huggingface langchain-text-splitters sentence-transformers

In [ ]:
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv(override=True)

In [ ]:
def get_neo4j_driver():
    """Connect to Neo4j."""

    uri = os.getenv("NEO4J_URI")
    username = os.getenv("NEO4J_USERNAME")
    password = os.getenv("NEO4J_PASSWORD")

    return GraphDatabase.driver(
        uri,
        auth=(username, password)
    )


driver = get_neo4j_driver()

driver.verify_connectivity()

print("Connected to Neo4j!")

In [ ]:
teachers = [
    {
        "name": "Daniel Smith",
        "department": "Science",
        "head": False
    },
    {
        "name": "Rachel Wilson",
        "department": "Science",
        "head": True
    },
    {
        "name": "Laura Taylor",
        "department": "Mathematics",
        "head": True
    },
    {
        "name": "James Miller",
        "department": "English",
        "head": True
    },
    {
        "name": "Priya Patel",
        "department": "Social Studies",
        "head": True
    },
    {
        "name": "Michael Brown",
        "department": "Mathematics",
        "head": False
    }
]

In [ ]:
classes = [
    {
        "class": "Physics 101",
        "subject": "Physics",
        "teacher": "Daniel Smith"
    },
    {
        "class": "Biology 101",
        "subject": "Biology",
        "teacher": "Rachel Wilson"
    },
    {
        "class": "Algebra 2",
        "subject": "Mathematics",
        "teacher": "Laura Taylor"
    },
    {
        "class": "Geometry",
        "subject": "Mathematics",
        "teacher": "Michael Brown"
    },
    {
        "class": "English Literature",
        "subject": "English",
        "teacher": "James Miller"
    },
    {
        "class": "US History",
        "subject": "History",
        "teacher": "Priya Patel"
    }
]

In [ ]:
students = [
    {"name": "Emma Johnson", "classes": ["Physics 101", "Algebra 2"]},
    {"name": "Noah Williams", "classes": ["Physics 101", "English Literature"]},
    {"name": "Sophia Davis", "classes": ["Biology 101", "Algebra 2"]},
    {"name": "Liam Anderson", "classes": ["Physics 101", "US History"]},
    {"name": "Olivia Martin", "classes": ["Biology 101", "Geometry"]},
    {"name": "Ethan Thompson", "classes": ["Physics 101", "Geometry"]},
    {"name": "Ava Garcia", "classes": ["Biology 101", "English Literature"]},
    {"name": "Lucas Martinez", "classes": ["Algebra 2", "US History"]},
    {"name": "Mia Robinson", "classes": ["Physics 101", "English Literature"]},
    {"name": "Benjamin Clark", "classes": ["Biology 101", "Geometry"]},
    {"name": "Charlotte Lewis", "classes": ["Algebra 2", "English Literature"]},
    {"name": "Henry Walker", "classes": ["Physics 101", "US History"]},
    {"name": "Amelia Hall", "classes": ["Biology 101", "Algebra 2"]},
    {"name": "Alexander Young", "classes": ["Geometry", "US History"]},
    {"name": "Harper King", "classes": ["Physics 101", "English Literature"]},
    {"name": "Daniel Wright", "classes": ["Biology 101", "Geometry"]},
    {"name": "Evelyn Scott", "classes": ["Algebra 2", "US History"]},
    {"name": "Matthew Green", "classes": ["Physics 101", "Geometry"]},
    {"name": "Abigail Baker", "classes": ["Biology 101", "English Literature"]},
    {"name": "Jackson Adams", "classes": ["Algebra 2", "US History"]}
]

In [ ]:
def add_teachers(driver, teachers):
    """Create teachers and departments."""

    query = """
    MERGE (t:Teacher {name: $name})
    MERGE (d:Department {name: $department})

    MERGE (t)-[:BELONGS_TO]->(d)

    FOREACH (_ IN CASE WHEN $head THEN [1] ELSE [] END |
        MERGE (t)-[:HEADS]->(d)
    )
    """

    with driver.session() as session:
        for teacher in teachers:
            session.run(query, **teacher)

    print("Teachers and departments added!")

In [ ]:
add_teachers(driver, teachers)

In [ ]:
def add_classes(driver, classes):
    """Create classes, subjects and teaching relationships."""

    query = """
    MATCH (t:Teacher {name: $teacher})

    MERGE (c:Class {name: $class})
    MERGE (s:Subject {name: $subject})

    MERGE (t)-[:TEACHES]->(c)
    MERGE (c)-[:FOR_SUBJECT]->(s)
    """

    with driver.session() as session:
        for item in classes:
            session.run(query, **item)

    print("Classes and subjects added!")

In [ ]:
add_classes(driver, classes)

In [ ]:
def add_students(driver, students):
    """Create students and enroll them in classes."""

    query = """
    MERGE (s:Student {name: $student})

    WITH s

    UNWIND $classes AS class_name

    MATCH (c:Class {name: class_name})

    MERGE (s)-[:ENROLLED_IN]->(c)
    """

    with driver.session() as session:
        for student in students:
            session.run(
                query,
                student=student["name"],
                classes=student["classes"]
            )

    print("Students added!")

In [ ]:
add_students(driver, students)

In [ ]:
def show_relationships(driver):
    """Show graph relationships."""

    query = """
    MATCH (a)-[r]->(b)

    RETURN
        a.name AS source,
        type(r) AS relationship,
        b.name AS target

    LIMIT 20
    """

    with driver.session() as session:
        results = session.run(query)

        for record in results:
            print(
                record["source"],
                "--",
                record["relationship"],
                "-->",
                record["target"]
            )

In [ ]:
show_relationships(driver)

In [ ]:
school_documents = [
    {
        "title": "Late Assignment Policy",
        "category": "Policy",
        "text": """
        Students may submit assignments up to five school days late.
        Assignments submitted one or two days late receive no penalty.
        After the second day, a 10 percent deduction is applied.
        Teachers may approve extensions for illness or other documented circumstances.
        """
    },
    {
        "title": "Attendance Policy",
        "category": "Policy",
        "text": """
        Students are expected to attend all scheduled classes.
        Parents or guardians must report an absence before 9 AM.
        Students with more than five unexcused absences in a semester
        must meet with their counselor and parent or guardian.
        """
    },
    {
        "title": "Grading Policy",
        "category": "Policy",
        "text": """
        Greenwood Academy uses a standard letter grading system.
        An A represents scores from 90 to 100 percent.
        A B represents scores from 80 to 89 percent.
        A C represents scores from 70 to 79 percent.
        Students scoring below 70 percent may be offered additional academic support.
        """
    },
    {
        "title": "Science Lab Incident",
        "category": "Incident",
        "text": """
        On September 12, a water leak affected Science Lab 3.
        Physics 101 was temporarily moved to Science Lab 5.
        No students or teachers were injured.
        Maintenance repaired the leak that afternoon.
        Physics 101 returned to Science Lab 3 the following day.
        """
    },
    {
        "title": "Field Trip Policy",
        "category": "Policy",
        "text": """
        Students participating in school field trips must submit
        a signed parent or guardian permission form.
        Permission forms must be submitted at least three school days
        before the trip.
        Students without a completed permission form cannot participate.
        """
    },
    {
        "title": "Technology Policy",
        "category": "Policy",
        "text": """
        Students may use school laptops for academic purposes.
        Personal gaming and unauthorized software installation are prohibited.
        Students must report lost or damaged devices to the technology office.
        """
    }
]

In [ ]:
from langchain_core.documents import Document


documents = []

for item in school_documents:

    doc = Document(
        page_content=item["text"],
        metadata={
            "title": item["title"],
            "category": item["category"]
        }
    )

    documents.append(doc)

In [ ]:
chunks = documents.copy()

print("Documents:", len(documents))
print("Chunks:", len(chunks))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings


embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
for chunk in chunks:

    chunk.metadata["embedding"] = embedding_model.embed_query(
        chunk.page_content
    )

In [ ]:
def store_chunks(driver, chunks):
    """Store school document chunks in Neo4j."""

    query = """
    MERGE (c:Chunk {chunkId: $chunk_id})

    SET c.text = $text,
        c.title = $title,
        c.category = $category,
        c.embedding = $embedding
    """

    with driver.session() as session:

        for i, chunk in enumerate(chunks):

            session.run(
                query,
                chunk_id=f"chunk_{i}",
                text=chunk.page_content,
                title=chunk.metadata["title"],
                category=chunk.metadata["category"],
                embedding=chunk.metadata["embedding"]
            )

    print("Chunks stored!")

In [ ]:
store_chunks(driver, chunks)

In [ ]:
def create_vector_index(driver):
    """Create vector index for school document chunks."""

    query = """
    CREATE VECTOR INDEX school_chunks IF NOT EXISTS

    FOR (c:Chunk)
    ON c.embedding

    OPTIONS {
        indexConfig: {
            `vector.dimensions`: 384,
            `vector.similarity_function`: 'cosine'
        }
    }
    """

    with driver.session() as session:
        session.run(query)

    print("Vector index created!")

In [ ]:
create_vector_index(driver)

In [ ]:
def create_fulltext_index(driver):
    """Create full-text index for student and teacher names."""

    query = """
    CREATE FULLTEXT INDEX school_names IF NOT EXISTS

    FOR (n:Student|Teacher)
    ON EACH [n.name]
    """

    with driver.session() as session:
        session.run(query)

    print("Full-text index created!")

In [ ]:
create_fulltext_index(driver)

In [ ]:
def search_name(driver, name):
    """Search students and teachers by name."""

    query = """
    CALL db.index.fulltext.queryNodes(
        'school_names',
        $name
    )
    YIELD node, score

    RETURN
        node.name AS name,
        labels(node) AS type,
        score

    LIMIT 5
    """

    with driver.session() as session:

        results = session.run(
            query,
            name=name
        )

        return [record.data() for record in results]

In [ ]:
from langchain_groq import ChatGroq


llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    timeout=60,
    max_retries=3
)

In [ ]:
def route_question(question):
    """Decide which retrieval method to use."""

    prompt = f"""
You are a router for a school knowledge system.

Choose exactly ONE retrieval method:
GRAPH, VECTOR, or EXACT.


GRAPH:
Use when the question asks about relationships between
students, teachers, classes, subjects, or departments.

Examples:
- Who teaches Emma Johnson?
- What department is Daniel Smith part of?
- Which classes does Emma Johnson attend?
- Who heads the department Daniel Smith belongs to?


VECTOR:
Use when the answer comes from school documents such as
policies, rules, incidents, events, causes, consequences,
or historical details.

Use VECTOR even if the question mentions a student,
teacher, or class when the requested information is about
an incident, policy, event, or document detail.

Examples:
- Why was Physics 101 moved?
- Where was Physics 101 moved after the water leak?
- What happened after the Science Lab 3 leak?
- Can homework be submitted late?


EXACT:
Use ONLY when the user wants to find, search for,
look up, or identify a student or teacher by name.

Do NOT use EXACT when the user asks about that person's
classes, department, teacher, subject, or other relationships.

Examples:
- Find Emma Johnson
- Search for Daniel Smith
- Look up Priya Patel


Question:
{question}

Return only one word:
GRAPH, VECTOR or EXACT
"""

    response = llm.invoke(prompt)

    return response.content.strip().upper()

In [ ]:
graph_schema = """
Nodes:

Student(name)
Examples: Emma Johnson, Noah Williams, Sophia Davis

Teacher(name)
Examples: Daniel Smith, Rachel Wilson, Laura Taylor, Michael Brown

Class(name)
Examples: Physics 101, Biology 101, Algebra 2,
Geometry, English Literature, US History

Subject(name)
Examples: Physics, Biology, Mathematics, English, History

Department(name)
Examples: Science, Mathematics, English, Social Studies


Relationships:

Student -[:ENROLLED_IN]-> Class

Teacher -[:TEACHES]-> Class

Class -[:FOR_SUBJECT]-> Subject

Teacher -[:BELONGS_TO]-> Department

Teacher -[:HEADS]-> Department
"""

In [ ]:
## when the router routes the question to GRAPH, the LLM will use the above graph
## schema and create a CYPHER query to query NEO4J
def generate_cypher(question):
    """Convert a graph question into Cypher."""

    prompt = f"""
You are a Neo4j Cypher expert.

Use only the graph schema and example values below.

{graph_schema}

Rules:
- Generate exactly one read-only Cypher query.
- Use only nodes and relationships from the schema.
- Identify whether each named entity is a Student, Teacher,
  Class, Subject, or Department using the example values.
- Follow relationships exactly as defined in the schema.
- Support multi-hop relationship questions.
- Always give returned values a clear alias.
- Never return an empty response.
- Return only the Cypher query.
- Do not include markdown or explanations.

Examples:

Question:
Who teaches Geometry?

Cypher:
MATCH (t:Teacher)-[:TEACHES]->(c:Class {{name: "Geometry"}})
RETURN t.name AS teacher


Question:
Who heads the department that Daniel Smith belongs to?

Cypher:
MATCH (t:Teacher {{name: "Daniel Smith"}})
-[:BELONGS_TO]->(d:Department)
<-[:HEADS]-(h:Teacher)
RETURN h.name AS department_head


Question:
{question}

Cypher:
"""

    response = llm.invoke(prompt)

    return response.content.strip()

In [ ]:
def run_cypher(driver, cypher):
    """Run Cypher against Neo4j."""

    with driver.session() as session:

        results = session.run(cypher)

        return [record.data() for record in results]

In [ ]:
def vector_search(driver, question):
    """Search school documents semantically."""

    question_embedding = embedding_model.embed_query(
        question
    )

    query = """
    CALL db.index.vector.queryNodes(
        'school_chunks',
        3,
        $embedding
    )
    YIELD node, score

    RETURN
        node.text AS text,
        node.title AS title,
        score
    """

    with driver.session() as session:

        results = session.run(
            query,
            embedding=question_embedding
        )

        return [record.data() for record in results]

                    QUESTION
                       │
                       ▼
                     ROUTER
                       │
        ┌──────────────┼──────────────┐
        ▼              ▼              ▼
      GRAPH          VECTOR          EXACT
        │              │              │
     Cypher         Embedding      Full-text
        ↓              ↓              ↓
     Neo4j           Neo4j          Neo4j
        │              │              │
        ▼              ▼              ▼
 Graph results     Text chunks      Entity

In [ ]:
def extract_name(question):
    """Extract student or teacher name."""

    prompt = f"""
Extract the student or teacher name from the question.

Question:
{question}

Return only the person's name.
"""

    response = llm.invoke(prompt)

    return response.content.strip()

In [ ]:
def retrieve(question):
    """Route the question and retrieve relevant information."""

    route = route_question(question)

    print("Route:", route)

    if route == "GRAPH":

        cypher = generate_cypher(question)

        print("\nCypher:")
        print(cypher)

        try:
            results = run_cypher(
                driver,
                cypher
            )
        except Exception as e:
            print("Cypher failed:", e)
            results = []

    elif route == "VECTOR":

        results = vector_search(
            driver,
            question
        )

    elif route == "EXACT":

        name = extract_name(question)

        results = search_name(
            driver,
            name
        )

    else:
        results = []

    return route, results

In [ ]:
def generate_answer(question, results, route):
    """Generate answer from retrieved results."""

    if not results:
        return "I don't have enough information."

    if route == "GRAPH":

        instructions = """
The retrieved information contains structured graph query results.
Treat the returned fields as facts retrieved specifically for the user's question.
Answer directly using those results.
"""

    elif route == "VECTOR":

        instructions = """
The retrieved information contains text from school documents.

Answer only what the user asked.

Use the minimum information needed to give a complete and correct answer.

Include conditions, exceptions, penalties, requirements,
or limitations only when they directly affect the answer.

Do not summarize the entire retrieved document.
Do not include unrelated details even if they appear in the retrieved information.
"""

    elif route == "EXACT":

        instructions = """
The retrieved information contains the result of a person lookup.
Use it to identify whether the person is a student or teacher.
"""

    prompt = f"""
Answer the user's question using only the retrieved information.

Question:
{question}

Retrieved Information:
{results}

Instructions:
{instructions}

Do not add information that is not supported by the retrieved information.
Give a clear and concise answer.
"""

    response = llm.invoke(prompt)

    return response.content

In [ ]:
def ask_school_bot(question):
    """Complete School GraphRAG pipeline."""

    route, results = retrieve(question)

    answer = generate_answer(
        question,
        results,
        route
    )

    return answer

In [ ]:
def evaluate_question(question):
    """Return retrieval details and final answer for evaluation."""

    route, results = retrieve(question)

    answer = generate_answer(
        question,
        results,
        route
    )

    return {
        "route": route,
        "retrieved": results,
        "answer": answer
    }

### ROUTER EVALUATION

In [ ]:
import json

with open(
    "evaluation/datasets/router_eval.json",
    "r"
) as f:
    router_eval_data = json.load(f)

In [ ]:
# Run each of the above questions into the router to check how the router 
# behaves

router_results = []

for item in router_eval_data:

    actual = route_question(
        item["question"]
    )

    correct = actual == item["expected"]

    router_results.append({
        "question": item["question"],
        "expected": item["expected"],
        "actual": actual,
        "correct": correct
    })

In [ ]:
for result in router_results:

    print("\nQuestion:", result["question"])
    print("Expected:", result["expected"])
    print("Actual:", result["actual"])
    print("Correct:", result["correct"])

In [ ]:
correct_count = sum(
    result["correct"]
    for result in router_results
)

total = len(router_results)

accuracy = correct_count / total

print("Correct:", correct_count)
print("Total:", total)
print("Router Accuracy:", accuracy)

In [ ]:
%pip install pandas

In [ ]:
import pandas as pd

router_df = pd.DataFrame(router_results)

router_df.to_csv(
    "evaluation/router_eval_results.csv",
    index=False
)

router_df

## Vector Retrieval Evaluation

In [ ]:
import json

with open(
    "evaluation/datasets/vector_eval.json",
    "r"
) as f:
    vector_eval_data = json.load(f)

In [ ]:
vector_eval_results = []

for item in vector_eval_data:

    results = vector_search(
        driver,
        item["question"]
    )

    retrieved_titles = [
        result["title"]
        for result in results
    ]

    vector_eval_results.append({
        "question": item["question"],
        "expected": item["expected"],
        "retrieved": retrieved_titles,
        "hit@3": item["expected"] in retrieved_titles
    })

In [ ]:
for result in vector_eval_results:

    print("\nQuestion:", result["question"])
    print("Expected:", result["expected"])
    print("Retrieved:", result["retrieved"])

In [ ]:
hits = sum(
    result["hit@3"]
    for result in vector_eval_results
)

hit_rate_3 = hits / len(vector_eval_results)

print("Hits:", hits)
print("Total:", len(vector_eval_results))
print("Hit@3:", hit_rate_3)

In [ ]:
for result in vector_eval_results:

    expected = result["expected"]
    retrieved = result["retrieved"]

    if expected in retrieved:

        rank = retrieved.index(expected) + 1

        reciprocal_rank = 1 / rank

    else:

        rank = None
        reciprocal_rank = 0

    result["rank"] = rank
    result["reciprocal_rank"] = reciprocal_rank

In [ ]:
for result in vector_eval_results:

    print("\nQuestion:", result["question"])
    print("Expected:", result["expected"])
    print("Rank:", result["rank"])
    print("Reciprocal Rank:", result["reciprocal_rank"])

In [ ]:
## Save VECTOR retrieval EVAL results to CSV

vector_df = pd.DataFrame(vector_eval_results)

vector_df.to_csv(
    "evaluation/vector_eval_results.csv",
    index=False
)

display(vector_df)

## Graph Retrieval Evaluation

In [ ]:
import json

with open(
    "evaluation/datasets/graph_eval.json",
    "r"
) as f:
    graph_eval_data = json.load(f)

In [ ]:
def extract_graph_values(results):
    """Extract values from Neo4j results."""

    values = []

    for record in results:
        for value in record.values():
            values.append(value)

    return values

In [ ]:
for item in graph_eval_data:

    cypher = generate_cypher(item["question"])

    print("\nQuestion:", item["question"])
    print("Cypher:", repr(cypher))

In [ ]:
graph_eval_results = []

for item in graph_eval_data:

    cypher = generate_cypher(
        item["question"]
    )

    if not cypher:

        actual = []
        correct = False

    else:

        results = run_cypher(
            driver,
            cypher
        )

        actual = extract_graph_values(results)

        correct = set(actual) == set(item["expected"])

    graph_eval_results.append({
        "question": item["question"],
        "expected": item["expected"],
        "actual": actual,
        "correct": correct,
        "cypher": cypher
    })

In [ ]:
correct_count = sum(
    result["correct"]
    for result in graph_eval_results
)

total = len(graph_eval_results)

graph_accuracy = correct_count / total

print("Correct:", correct_count)
print("Total:", total)
print("Graph Retrieval Accuracy:", graph_accuracy)

In [ ]:
for result in graph_eval_results:

    if not result["correct"]:
        print("\nQuestion:", result["question"])
        print("Expected:", result["expected"])
        print("Actual:", result["actual"])
        print("Cypher:", repr(result["cypher"]))

In [ ]:
graph_df = pd.DataFrame(graph_eval_results)

graph_df.to_csv(
    "evaluation/graph_eval_baseline.csv",
    index=False
)

display(graph_df)

In [ ]:
evaluation_summary = pd.DataFrame([
    {
        "Component": "Router",
        "Metric": "Accuracy",
        "Score": 1.0,
        "Questions": 15
    },
    {
        "Component": "Vector Retrieval",
        "Metric": "Hit@3",
        "Score": hit_rate_3,
        "Questions": 10
    },
    {
        "Component": "Vector Retrieval",
        "Metric": "MRR",
        "Score": sum(r["reciprocal_rank"] for r in vector_eval_results) / len(vector_eval_results),
        "Questions": 10
    },
    {
        "Component": "Graph Retrieval - Baseline",
        "Metric": "Accuracy",
        "Score": graph_accuracy,
        "Questions": 10
    }
])

display(evaluation_summary)

In [ ]:
graph_improved_df = pd.DataFrame(graph_eval_results)

graph_improved_df.to_csv(
    "evaluation/graph_eval_improved.csv",
    index=False
)

display(graph_improved_df)

In [ ]:
new_result = pd.DataFrame([
    {
        "Component": "Graph Retrieval - Improved",
        "Metric": "Accuracy",
        "Score": graph_accuracy,
        "Questions": total
    }
])

evaluation_summary = pd.concat(
    [evaluation_summary, new_result],
    ignore_index=True
)

display(evaluation_summary)

###     EXACT/full-text retrieval

In [ ]:
import json

with open(
    "evaluation/datasets/exact_eval.json",
    "r"
) as f:
    exact_eval_data = json.load(f)

In [ ]:
exact_eval_results = []

for item in exact_eval_data:

    results = search_name(
        driver,
        item["query"]
    )

    retrieved_names = [
        result["name"]
        for result in results
    ]

    correct = item["expected"] in retrieved_names

    exact_eval_results.append({
        "query": item["query"],
        "expected": item["expected"],
        "actual": retrieved_names,
        "correct": correct
    })

In [ ]:
for result in exact_eval_results:

    print("\nQuery:", result["query"])
    print("Expected:", result["expected"])
    print("Actual:", result["actual"])
    print("Correct:", result["correct"])

In [ ]:
correct_count = sum(
    result["correct"]
    for result in exact_eval_results
)

total = len(exact_eval_results)

exact_accuracy = correct_count / total

print("Correct:", correct_count)
print("Total:", total)
print("Exact Retrieval Accuracy:", exact_accuracy)

In [ ]:
exact_df = pd.DataFrame(exact_eval_results)

exact_df.to_csv(
    "evaluation/exact_eval_results.csv",
    index=False
)

display(exact_df)

In [ ]:
new_result = pd.DataFrame([
    {
        "Component": "Exact Retrieval",
        "Version": "-",
        "Metric": "Accuracy",
        "Score": exact_accuracy,
        "Questions": total
    }
])

evaluation_summary = pd.concat(
    [evaluation_summary, new_result],
    ignore_index=True
)

display(evaluation_summary)

In [ ]:
evaluation_summary.to_csv(
    "evaluation/evaluation_summary.csv",
    index=False
)

###     EVALUATE GENERATION STEP    ###

In [ ]:
import json

with open(
    "evaluation/datasets/generation_eval.json",
    "r"
) as f:
    generation_eval_data = json.load(f)

In [ ]:
def evaluate_generation(question):
    """Return retrieval details and final answer for evaluation."""

    route, results = retrieve(question)

    answer = generate_answer(
        question,
        results,
        route
    )

    return {
        "route": route,
        "retrieved": results,
        "answer": answer
    }

In [ ]:
generation_results = []

for item in generation_eval_data:

    result = evaluate_generation(
        item["question"]
    )

    generation_results.append({
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "route": result["route"],
        "retrieved": result["retrieved"],
        "generated_answer": result["answer"]
    })

In [ ]:
for result in generation_results:

    print("\nQuestion:", result["question"])
    print("Expected:", result["expected_answer"])
    print("Route:", result["route"])
    print("Retrieved:", result["retrieved"])
    print("Generated:", result["generated_answer"])

In [ ]:
generation_df = pd.DataFrame(generation_results)

generation_df["retrieved"] = generation_df["retrieved"].apply(
    json.dumps
)

generation_df.to_csv(
    "evaluation/generation_eval_results.csv",
    index=False
)

display(generation_df)

In [ ]:
def judge_correctness(question, expected, generated):
    """Judge whether the generated answer is correct."""

    prompt = f"""
You are evaluating the correctness of an answer.

Question:
{question}

Expected Answer:
{expected}

Generated Answer:
{generated}

Rules:

PASS if:
- The generated answer contains all essential information
  from the expected answer needed to answer the question.
- Different wording is allowed.
- A shorter answer is allowed if it fully answers the question.

FAIL if:
- An essential fact is missing.
- The answer contains an incorrect fact.
- The answer contradicts the expected answer.
- The answer does not answer the question.

Return only:
PASS or FAIL
"""

    response = llm.invoke(prompt)

    return response.content.strip().upper()

In [ ]:
for result in generation_results:

    result["correctness"] = judge_correctness(
        result["question"],
        result["expected_answer"],
        result["generated_answer"]
    )

In [ ]:
for result in generation_results:

    if result["correctness"] == "FAIL":

        print("\nQuestion:", result["question"])
        print("Expected:", result["expected_answer"])
        print("Generated:", result["generated_answer"])

In [ ]:
for result in generation_results:

    if result["correctness"] == "FAIL":

        print("\nQUESTION:")
        print(result["question"])

        print("\nRETRIEVED:")
        print(result["retrieved"])

        print("\nGENERATED:")
        print(result["generated_answer"])

        print("-" * 50)

In [ ]:
correct_count = sum(
    result["correctness"] == "PASS"
    for result in generation_results
)

total = len(generation_results)

correctness_rate = correct_count / total

print("Correct:", correct_count)
print("Total:", total)
print("Generation Correctness Rate:", correctness_rate)

In [ ]:
generation_judged_df = pd.DataFrame(generation_results)

generation_judged_df["retrieved"] = generation_judged_df["retrieved"].apply(
    json.dumps
)

generation_judged_df.to_csv(
    "evaluation/generation_eval_baseline_judged.csv",
    index=False
)

display(generation_judged_df)

In [ ]:
new_result = pd.DataFrame([
    {
        "Component": "Generation",
        "Version": "Baseline",
        "Metric": "Correctness",
        "Score": correctness_rate,
        "Questions": total
    }
])

evaluation_summary = pd.concat(
    [evaluation_summary, new_result],
    ignore_index=True
)

display(evaluation_summary)

In [ ]:
evaluation_summary.to_csv(
    "evaluation/evaluation_summary.csv",
    index=False
)

In [ ]:
generation_improved_results = []

for item in generation_eval_data:

    result = evaluate_generation(
        item["question"]
    )

    generation_improved_results.append({
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "route": result["route"],
        "retrieved": result["retrieved"],
        "generated_answer": result["answer"]
    })

In [ ]:
for result in generation_improved_results:

    if result["question"] in [
        "Can I submit my homework four days late?",
        "What happens if I have more than five unexcused absences?"
    ]:

        print("\nQuestion:", result["question"])
        print("Retrieved:", result["retrieved"])
        print("Generated:", result["generated_answer"])

In [ ]:
generation_chunking_df = pd.DataFrame(
    generation_improved_results
)

generation_chunking_df["retrieved"] = (
    generation_chunking_df["retrieved"].apply(json.dumps)
)

generation_chunking_df.to_csv(
    "evaluation/generation_eval_chunking_improved.csv",
    index=False
)

display(generation_chunking_df)

In [ ]:
for result in generation_improved_results:

    result["correctness"] = judge_correctness(
        result["question"],
        result["expected_answer"],
        result["generated_answer"]
    )

In [ ]:
correct_count = sum(
    result["correctness"] == "PASS"
    for result in generation_improved_results
)

total = len(generation_improved_results)

chunking_correctness_rate = correct_count / total

print("Correct:", correct_count)
print("Total:", total)
print(
    "Correctness after chunking improvement:",
    chunking_correctness_rate
)

In [ ]:
generation_chunking_judged_df = pd.DataFrame(
    generation_improved_results
)

generation_chunking_judged_df["retrieved"] = (
    generation_chunking_judged_df["retrieved"].apply(json.dumps)
)

generation_chunking_judged_df.to_csv(
    "evaluation/generation_eval_chunking_improved_judged.csv",
    index=False
)

In [ ]:
new_result = pd.DataFrame([
    {
        "Component": "Generation",
        "Version": "Chunking Improved",
        "Metric": "Correctness",
        "Score": chunking_correctness_rate,
        "Questions": total
    }
])

evaluation_summary = pd.concat(
    [evaluation_summary, new_result],
    ignore_index=True
)

display(evaluation_summary)

In [ ]:
generation_prompt_improved_results = []

for item in generation_eval_data:

    result = evaluate_generation(
        item["question"]
    )

    generation_prompt_improved_results.append({
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "route": result["route"],
        "retrieved": result["retrieved"],
        "generated_answer": result["answer"]
    })

In [ ]:
for result in generation_prompt_improved_results:

    result["correctness"] = judge_correctness(
        result["question"],
        result["expected_answer"],
        result["generated_answer"]
    )

In [ ]:
for result in generation_prompt_improved_results:

    if result["correctness"] == "FAIL":

        print("\nQuestion:", result["question"])
        print("Expected:", result["expected_answer"])
        print("Retrieved:", result["retrieved"])
        print("Generated:", result["generated_answer"])

In [ ]:
correct_count = sum(
    result["correctness"] == "PASS"
    for result in generation_prompt_improved_results
)

total = len(generation_prompt_improved_results)

prompt_improved_correctness_rate = (
    correct_count / total
)

print("Correct:", correct_count)
print("Total:", total)
print(
    "Generation Correctness Rate:",
    prompt_improved_correctness_rate
)

In [ ]:
prompt_v1_df = pd.DataFrame(
    generation_prompt_improved_results
)

prompt_v1_df["retrieved"] = (
    prompt_v1_df["retrieved"].apply(json.dumps)
)

prompt_v1_df.to_csv(
    "evaluation/generation_eval_prompt_v1.csv",
    index=False
)

In [ ]:
new_result = pd.DataFrame([
    {
        "Component": "Generation",
        "Version": "Prompt v1",
        "Metric": "Correctness",
        "Score": prompt_improved_correctness_rate,
        "Questions": total
    }
])

evaluation_summary = pd.concat(
    [evaluation_summary, new_result],
    ignore_index=True
)

evaluation_summary.to_csv(
    "evaluation/evaluation_summary.csv",
    index=False
)

In [ ]:
generation_route_aware_results = []

for item in generation_eval_data:

    result = evaluate_question(
        item["question"]
    )

    generation_route_aware_results.append({
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "route": result["route"],
        "retrieved": result["retrieved"],
        "generated_answer": result["answer"]
    })

In [ ]:
for result in generation_route_aware_results:

    result["correctness"] = judge_correctness(
        result["question"],
        result["expected_answer"],
        result["generated_answer"]
    )

In [ ]:
correct_count = sum(
    result["correctness"] == "PASS"
    for result in generation_route_aware_results
)

total = len(generation_route_aware_results)

route_aware_correctness_rate = correct_count / total

print("Correct:", correct_count)
print("Total:", total)
print(
    "Route-Aware Generation Correctness:",
    route_aware_correctness_rate
)

In [ ]:
for result in generation_route_aware_results:

    if result["correctness"] == "FAIL":

        print("\nQuestion:")
        print(result["question"])

        print("\nExpected:")
        print(result["expected_answer"])

        print("\nRoute:")
        print(result["route"])

        print("\nRetrieved:")
        print(result["retrieved"])

        print("\nGenerated:")
        print(result["generated_answer"])

In [ ]:
route_aware_df = pd.DataFrame(
    generation_route_aware_results
)

route_aware_df["retrieved"] = (
    route_aware_df["retrieved"].apply(json.dumps)
)

route_aware_df.to_csv(
    "evaluation/generation_eval_route_aware.csv",
    index=False
)

display(route_aware_df)

In [ ]:
new_result = pd.DataFrame([
    {
        "Component": "Generation",
        "Version": "Route-Aware",
        "Metric": "Correctness",
        "Score": route_aware_correctness_rate,
        "Questions": total
    }
])

evaluation_summary = pd.concat(
    [evaluation_summary, new_result],
    ignore_index=True
)

display(evaluation_summary)

In [ ]:
with open(
    "evaluation/datasets/router_validation.json",
    "r"
) as f:
    router_validation_data = json.load(f)

In [ ]:
router_validation_results = []

for item in router_validation_data:

    actual = route_question(
        item["question"]
    )

    correct = actual == item["expected"]

    router_validation_results.append({
        "question": item["question"],
        "expected": item["expected"],
        "actual": actual,
        "correct": correct
    })

In [ ]:
correct_count = sum(
    result["correct"]
    for result in router_validation_results
)

total = len(router_validation_results)

router_validation_accuracy = (
    correct_count / total
)

print("Correct:", correct_count)
print("Total:", total)
print(
    "Router Validation Accuracy:",
    router_validation_accuracy
)

In [ ]:
for result in router_validation_results:

    if not result["correct"]:
        print("\nQuestion:", result["question"])
        print("Expected:", result["expected"])
        print("Actual:", result["actual"])

In [ ]:
router_validation_df = pd.DataFrame(
    router_validation_results
)

router_validation_df.to_csv(
    "evaluation/router_validation_baseline.csv",
    index=False
)

display(router_validation_df)

In [ ]:
new_result = pd.DataFrame([
    {
        "Component": "Router",
        "Version": "Validation Baseline",
        "Metric": "Accuracy",
        "Score": router_validation_accuracy,
        "Questions": total
    }
])

evaluation_summary = pd.concat(
    [evaluation_summary, new_result],
    ignore_index=True
)

evaluation_summary.to_csv(
    "evaluation/evaluation_summary.csv",
    index=False
)

In [ ]:
generation_final_results = []

for item in generation_eval_data:

    result = evaluate_question(
        item["question"]
    )

    generation_final_results.append({
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "route": result["route"],
        "retrieved": result["retrieved"],
        "generated_answer": result["answer"]
    })

In [ ]:
for result in generation_final_results:

    result["correctness"] = judge_correctness(
        result["question"],
        result["expected_answer"],
        result["generated_answer"]
    )

In [ ]:
for result in generation_final_results:

    if result["correctness"] == "FAIL":

        print("\nQuestion:", result["question"])
        print("Expected:", result["expected_answer"])
        print("Route:", result["route"])
        print("Retrieved:", result["retrieved"])
        print("Generated:", result["generated_answer"])

In [ ]:
correct_count = sum(
    result["correctness"] == "PASS"
    for result in generation_final_results
)

total = len(generation_final_results)

final_correctness_rate = correct_count / total

print("Correct:", correct_count)
print("Total:", total)
print(
    "Final Generation Correctness:",
    final_correctness_rate
)

In [ ]:
generation_final_df = pd.DataFrame(
    generation_final_results
)

generation_final_df["retrieved"] = (
    generation_final_df["retrieved"].apply(json.dumps)
)

generation_final_df.to_csv(
    "evaluation/generation_eval_final.csv",
    index=False
)

display(generation_final_df)

In [ ]:
new_result = pd.DataFrame([
    {
        "Component": "Generation",
        "Version": "Final",
        "Metric": "Correctness",
        "Score": final_correctness_rate,
        "Questions": total
    }
])

evaluation_summary = pd.concat(
    [evaluation_summary, new_result],
    ignore_index=True
)

evaluation_summary.to_csv(
    "evaluation/evaluation_summary.csv",
    index=False
)

display(evaluation_summary)

In [ ]:
def judge_faithfulness(question, retrieved, generated):
    """Judge whether the answer is supported by retrieved information."""

    prompt = f"""
You are evaluating the faithfulness of an answer.

Question:
{question}

Retrieved Information:
{retrieved}

Generated Answer:
{generated}

PASS if:
- Every factual claim in the generated answer is supported
  by the retrieved information.
- The answer may rephrase the retrieved information.
- The answer may contain fewer details than the retrieved information.

FAIL if:
- The answer contains any factual claim that is not supported
  by the retrieved information.
- The answer contradicts the retrieved information.
- The answer invents names, numbers, reasons, dates,
  relationships, conditions, or other facts.

Evaluate only faithfulness to the retrieved information.
Do not judge whether the answer is complete.

Return only:
PASS or FAIL
"""

    response = llm.invoke(prompt)

    return response.content.strip().upper()

In [ ]:
for result in generation_final_results:

    result["faithfulness"] = judge_faithfulness(
        result["question"],
        result["retrieved"],
        result["generated_answer"]
    )

In [ ]:
for result in generation_final_results:

    if result["faithfulness"] == "FAIL":

        print("\nQuestion:")
        print(result["question"])

        print("\nRetrieved:")
        print(result["retrieved"])

        print("\nGenerated:")
        print(result["generated_answer"])

In [ ]:
faithful_count = sum(
    result["faithfulness"] == "PASS"
    for result in generation_final_results
)

total = len(generation_final_results)

faithfulness_rate = faithful_count / total

print("Faithful:", faithful_count)
print("Total:", total)
print("Faithfulness Rate:", faithfulness_rate)

In [ ]:
generation_final_df = pd.DataFrame(
    generation_final_results
)

generation_final_df["retrieved"] = (
    generation_final_df["retrieved"].apply(json.dumps)
)

generation_final_df.to_csv(
    "evaluation/generation_eval_final.csv",
    index=False
)

display(generation_final_df)

In [ ]:
new_result = pd.DataFrame([
    {
        "Component": "Generation",
        "Version": "Final",
        "Metric": "Faithfulness",
        "Score": faithfulness_rate,
        "Questions": total
    }
])

evaluation_summary = pd.concat(
    [evaluation_summary, new_result],
    ignore_index=True
)

evaluation_summary.to_csv(
    "evaluation/evaluation_summary.csv",
    index=False
)

display(evaluation_summary)

In [ ]:
with open(
    "evaluation/datasets/final_validation.json",
    "r"
) as f:
    final_validation_data = json.load(f)

print("Final validation questions:", len(final_validation_data))

In [ ]:
final_validation_results = []

for item in final_validation_data:

    result = evaluate_question(
        item["question"]
    )

    final_validation_results.append({
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "route": result["route"],
        "retrieved": result["retrieved"],
        "generated_answer": result["answer"]
    })

In [ ]:
for result in final_validation_results:

    result["correctness"] = judge_correctness(
        result["question"],
        result["expected_answer"],
        result["generated_answer"]
    )

In [ ]:
for result in final_validation_results:

    result["faithfulness"] = judge_faithfulness(
        result["question"],
        result["retrieved"],
        result["generated_answer"]
    )

In [ ]:
print(final_validation_results[0].keys())

In [ ]:
correct_count = sum(
    result["correctness"] == "PASS"
    for result in final_validation_results
)

faithful_count = sum(
    result["faithfulness"] == "PASS"
    for result in final_validation_results
)

total = len(final_validation_results)

print("Total:", total)
print("Correct:", correct_count)
print("Correctness Rate:", correct_count / total)

print("Faithful:", faithful_count)
print("Faithfulness Rate:", faithful_count / total)

In [ ]:
for result in final_validation_results:

    if result["correctness"] == "":

        print("Question:")
        print(result["question"])

        print("\nExpected:")
        print(result["expected_answer"])

        print("\nRetrieved:")
        print(result["retrieved"])

        print("\nGenerated:")
        print(result["generated_answer"])

        print("\nFaithfulness:")
        print(result["faithfulness"])

In [ ]:
for result in final_validation_results:
    print(
        repr(result["correctness"]),
        "-",
        result["question"]
    )

In [ ]:
for result in final_validation_results:

    if result["correctness"] == "":

        print("Question:")
        print(result["question"])

        print("\nExpected:")
        print(result["expected_answer"])

        print("\nRetrieved:")
        print(result["retrieved"])

        print("\nGenerated:")
        print(result["generated_answer"])

        print("\nFaithfulness:")
        print(result["faithfulness"])

In [ ]:
for result in final_validation_results:

    if result["correctness"] == "":

        result["correctness"] = judge_correctness(
            result["question"],
            result["expected_answer"],
            result["generated_answer"]
        )

        print("New judgment:", repr(result["correctness"]))

In [ ]:
correct_count = sum(
    result["correctness"] == "PASS"
    for result in final_validation_results
)

total = len(final_validation_results)

print("Correct:", correct_count)
print("Total:", total)
print("Correctness Rate:", correct_count / total)

In [ ]:
final_validation_df = pd.DataFrame(
    final_validation_results
)

final_validation_df["retrieved"] = (
    final_validation_df["retrieved"].apply(json.dumps)
)

final_validation_df.to_csv(
    "evaluation/final_validation_results.csv",
    index=False
)

display(final_validation_df)

In [ ]:
final_summary = pd.DataFrame([
    {
        "Component": "Final Validation",
        "Version": "Unseen",
        "Metric": "Correctness",
        "Score": 1.0,
        "Questions": 15
    },
    {
        "Component": "Final Validation",
        "Version": "Unseen",
        "Metric": "Faithfulness",
        "Score": 1.0,
        "Questions": 15
    }
])

evaluation_summary = pd.concat(
    [evaluation_summary, final_summary],
    ignore_index=True
)

evaluation_summary.to_csv(
    "evaluation/evaluation_summary.csv",
    index=False
)

display(evaluation_summary)

In [ ]:
generation_precise_results = []

for item in generation_eval_data:

    result = evaluate_question(
        item["question"]
    )

    generation_precise_results.append({
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "route": result["route"],
        "retrieved": result["retrieved"],
        "generated_answer": result["answer"]
    })

In [288]:
import time

start_index = len(generation_precise_results)

print("Already completed:", start_index)

for item in generation_eval_data[start_index:]:

    result = evaluate_question(
        item["question"]
    )

    generation_precise_results.append({
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "route": result["route"],
        "retrieved": result["retrieved"],
        "generated_answer": result["answer"]
    })

    time.sleep(2)

Already completed: 19
Route: EXACT


In [294]:
correct_count = sum(
    result["correctness"] == "PASS"
    for result in generation_precise_results
)

faithful_count = sum(
    result["faithfulness"] == "PASS"
    for result in generation_precise_results
)

total = len(generation_precise_results)

precise_correctness_rate = correct_count / total
precise_faithfulness_rate = faithful_count / total

print("Correct:", correct_count)
print("Faithful:", faithful_count)
print("Total:", total)

print(
    "Correctness Rate:",
    precise_correctness_rate
)

print(
    "Faithfulness Rate:",
    precise_faithfulness_rate
)

Correct: 20
Faithful: 20
Total: 20
Correctness Rate: 1.0
Faithfulness Rate: 1.0


In [297]:
precision_summary = pd.DataFrame([
    {
        "Component": "Generation",
        "Version": "Precision Final",
        "Metric": "Correctness",
        "Score": precise_correctness_rate,
        "Questions": total
    },
    {
        "Component": "Generation",
        "Version": "Precision Final",
        "Metric": "Faithfulness",
        "Score": precise_faithfulness_rate,
        "Questions": total
    }
])

evaluation_summary = pd.concat(
    [evaluation_summary, precision_summary],
    ignore_index=True
)

evaluation_summary.to_csv(
    "evaluation/evaluation_summary.csv",
    index=False
)